# 🚀 Try Odibi in 5 Minutes

**No install. No setup. Just run.**

This notebook generates realistic simulated data from a YAML config — sensor readings with memory, drift, and mean reversion.

3 cells. That's it.

---

**What is Odibi?** A config-driven data pipeline framework. You describe *what* you want in YAML; Odibi handles *how*.

- 📄 [GitHub](https://github.com/henryodibi11/Odibi)
- 📚 [Docs](https://henryodibi11.github.io/Odibi/)

In [ ]:
# Cell 1: Install Odibi (takes ~30 seconds)
!pip install odibi altair -q
print("✅ Odibi installed!")

In [ ]:
# Cell 2: Define a simulation in YAML and run it
import logging
logging.getLogger("odibi").setLevel(logging.WARNING)

from pathlib import Path
from odibi.pipeline import PipelineManager

# Write a minimal simulation config
config_yaml = """
project: try_odibi
engine: pandas

connections:
  output:
    type: local
    base_path: ./data

story:
  connection: output
  path: stories/

system:
  connection: output

pipelines:
  - pipeline: demo
    nodes:
      - name: sensor_readings
        read:
          connection: null
          format: simulation
          options:
            simulation:
              scope:
                start_time: \"2026-01-01T00:00:00Z\"
                timestep: \"5m\"
                row_count: 200
                seed: 42
              entities:
                count: 3
                id_prefix: \"sensor_\"
              columns:
                - name: sensor_id
                  data_type: string
                  generator: {type: constant, value: \"{entity_id}\"}
                - name: timestamp
                  data_type: timestamp
                  generator: {type: timestamp}
                - name: temperature_c
                  data_type: float
                  generator:
                    type: random_walk
                    start: 22.0
                    min: 16.0
                    max: 30.0
                    volatility: 0.3
                    mean_reversion: 0.15
                    precision: 1
                - name: humidity_pct
                  data_type: float
                  generator:
                    type: range
                    min: 30.0
                    max: 70.0
                    distribution: normal
                    mean: 45.0
                    std_dev: 8.0
                - name: status
                  data_type: string
                  generator:
                    type: categorical
                    values: [Running, Idle, Error]
                    weights: [0.85, 0.12, 0.03]
        write:
          connection: output
          format: parquet
          path: bronze/sensors.parquet
          mode: overwrite
"""

Path("sim.yaml").write_text(config_yaml)
print("📄 Config written: sim.yaml")
print("⚡ Running pipeline...\n")

pm = PipelineManager.from_yaml("sim.yaml")
pm.run()

# Show what was generated
import pandas as pd
df = pd.read_parquet("data/bronze/sensors.parquet")
print(f"\n✅ Generated {len(df)} rows, {len(df.columns)} columns")
print(f"   Sensors: {sorted(df['sensor_id'].unique())}")
print(f"   Columns: {list(df.columns)}")
print(f"\n{df.head(10).to_string()}")

In [ ]:
# Cell 3: Visualize it
import altair as alt

alt.Chart(df).mark_line(opacity=0.7).encode(
    x=alt.X("timestamp:T", title="Time"),
    y=alt.Y("temperature_c:Q", title="Temperature (°C)"),
    color=alt.Color("sensor_id:N", title="Sensor")
).properties(
    title="Simulated Sensor Data — Generated from YAML",
    width=700,
    height=350
)

---

## That's it. YAML → Data → Chart.

What you just saw:
- **`random_walk`** with `mean_reversion` — temperature drifts realistically, not randomly
- **`categorical`** with `weights` — status values follow a real distribution (85% Running, 12% Idle, 3% Error)
- **`entities`** — 3 independent sensors, each with their own time series
- **Zero code changes needed** — swap `engine: pandas` for `engine: spark` and it runs on Databricks

## Want more?

The framework includes **38 simulation configs** across 7 industries:

| Domain | What it simulates |
|--------|-------------------|
| 🏢 Buildings | Occupancy sensors, HVAC feedback, CO2 drift |
| ⚙️ Compressors | Vibration, pressure surge, bearing wear |
| 🌡️ Cooling Towers | Approach temperature, efficiency, water balance |
| 🧪 Reactors | PID control, feed disturbance, exothermic reactions |
| 🚰 Wastewater | 5-stage treatment, storm surge, compliance |
| 🏭 Production | Machine maintenance, defect rates, shift patterns |
| 💰 Sales | Medallion architecture, category weights, order tiers |

**Advanced features:** `entity_overrides`, `scheduled_events`, `chaos` injection, `daily_profile`, `pid()`, `ema()`, `safe_div()`, `prev()`, `derive_columns`

---

⭐ **[Star the repo](https://github.com/henryodibi11/Odibi)** | 📚 **[Read the docs](https://henryodibi11.github.io/Odibi/)** | 🐛 **[Open an issue](https://github.com/henryodibi11/Odibi/issues)**